# TinyGlobalDet v2 inference

Chạy detector custom dưới 200k và fastText 3-gram, tạo `submission_tinyglobaldet.csv`.

In [ ]:
from pathlib import Path
import json,re,numpy as np,pandas as pd,torch,fasttext
from PIL import Image
from tinyglobaldet_model import TinyGlobalDet
ROOT=Path.cwd(); DATA=ROOT/'data'
if not DATA.exists():
    for p in ROOT.parents:
        if (p/'data').exists(): ROOT,DATA=p,p/'data'; break
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); SLOTS=2; NC=7; GW,GH=40,30
ckpt=ROOT/'runs'/'tinyglobaldet'/'best.pt'; ftpath=ROOT/'fasttext_work'/'intent_fasttext_3gram_200k.bin'; assert ckpt.exists(); assert ftpath.exists()
model=TinyGlobalDet(NC,SLOTS).to(DEVICE); model.load_state_dict(torch.load(ckpt,map_location=DEVICE)['model']); model.eval()
def nms(bs,ss,thr=.5):
    if not bs:return []
    b=torch.tensor(bs); s=torch.tensor(ss); order=s.argsort(descending=True); keep=[]
    while len(order):
        i=order[0].item(); keep.append(i)
        if len(order)==1:break
        xx1=torch.maximum(b[i,0],b[order[1:],0]); yy1=torch.maximum(b[i,1],b[order[1:],1]); xx2=torch.minimum(b[i,2],b[order[1:],2]); yy2=torch.minimum(b[i,3],b[order[1:],3]); inter=(xx2-xx1).clamp(min=0)*(yy2-yy1).clamp(min=0); ar=(b[:,2]-b[:,0])*(b[:,3]-b[:,1]); iou=inter/(ar[i]+ar[order[1:]]-inter+1e-7); order=order[1:][iou<=thr]
    return keep
@torch.no_grad()
def detect(path,conf=.2):
    im=Image.open(path).convert('RGB').resize((320,240)); x=torch.from_numpy(np.asarray(im,dtype=np.float32).transpose(2,0,1)/255.).unsqueeze(0).to(DEVICE); q=model(x)[0].sigmoid().cpu(); bs,ss,cs=[],[],[]
    for s in range(SLOTS):
        for gy in range(GH):
            for gx in range(GW):
                v=q[s,:,gy,gx]; sc,cl=v[5:].max(0); score=float(v[0]*sc)
                if score<conf:continue
                cx=(float(v[1])+gx)/GW; cy=(float(v[2])+gy)/GH; w=float(v[3]); h=float(v[4]); bs.append([(cx-w/2)*320,(cy-h/2)*240,(cx+w/2)*320,(cy+h/2)*240]); ss.append(score); cs.append(int(cl))
    keep=[]
    for c in set(cs):
        ids=[i for i,z in enumerate(cs) if z==c]; keep += [ids[j] for j in nms([bs[i] for i in ids],[ss[i] for i in ids])]
    return json.dumps([{'class_id':cs[i],'score':round(ss[i],6),'bbox':[round(max(0,min(320,z)),2) if j%2==0 else round(max(0,min(240,z)),2) for j,z in enumerate(bs[i])]} for i in keep],separators=(',',':'))
det={p.stem:detect(p) for p in sorted((DATA/'test'/'images').glob('*.png'))}; fm=fasttext.load_model(str(ftpath)); it=pd.read_csv(DATA/'intent_test.csv')
def clean(s):return re.sub(r'\s+',' ',re.sub(r'[^a-z0-9_\s]',' ',str(s).lower())).strip()
intent={r.question_id:fm.predict(clean(r.question),k=1)[0][0].replace('__label__','') for _,r in it.iterrows()}
sub=pd.read_csv(DATA/'sample_submission.csv'); sub['prediction']=sub.apply(lambda r:det[r.sample_id] if r.task_type=='DETECTION' else intent[r.sample_id],axis=1); out=ROOT/'submission_tinyglobaldet.csv'; sub.to_csv(out,index=False); print(out,sub.shape)